In [3]:
#!/usr/bin/env python3
"""
build_nigeria_power_topology.py

Power-only pipeline (final updated version):

 - Load power-sector assets (data/raw/power_sector_assets.csv) or fallback embedded BACKBONE_DATA
 - Normalise, deduplicate, snap to transmission lines (local preferred, OSM fallback)
 - Build provisional topology & compute betweenness centrality
 - Compute attacks-within-10km from Zenodo CSV (data/raw/combined_energy_attacks_02.csv)
 - Export GeoPackage and exposure CSV to data/processed/

Outputs:
  data/processed/nigeria_power_topology.gpkg  (layer 'power_nodes')
  data/processed/power_exposure_report.csv
"""
from pathlib import Path
import re
import sys
import logging
import io

import pandas as pd
import geopandas as gpd
import numpy as np
from shapely.geometry import Point
import networkx as nx

# Logging
logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")
log = logging.getLogger("nigeria_power_topology")

# -------------------
# Config
# -------------------
PROJECT_ROOT = Path(".")
DATA_DIR = PROJECT_ROOT / "data"
RAW_DIR = DATA_DIR / "raw"
PROC_DIR = DATA_DIR / "processed"
PROC_DIR.mkdir(parents=True, exist_ok=True)

POWER_CSV = RAW_DIR / "power_sector_assets.csv"   # primary input (power-only)
ZENODO_EVENTS = RAW_DIR / "combined_energy_attacks_02.csv"
LOCAL_LINES = RAW_DIR / "nigeria_transmission_lines.geojson"  # optional, preferred

OUTPUT_GPKG = PROC_DIR / "nigeria_power_topology.gpkg"
OUTPUT_LAYER = "power_nodes"
EXPOSURE_CSV = PROC_DIR / "power_exposure_report.csv"

CRS_WGS = "EPSG:4326"
CRS_METRIC = "EPSG:3857"

# -------------------
# Built-in fallback: full compiled asset list with coordinates filled
# Note: columns are:
# Asset_ID,Node_ID,Name,Type,Latitude,Longitude,Capacity,Units,State_Region,Source_URL,Confidence,Notes
# -------------------
BACKBONE_DATA = """Asset_ID,Node_ID,Name,Type,Latitude,Longitude,Capacity,Units,State_Region,Source_URL,Confidence,Notes
T_KAINJI,T_KAINJI,Kainji TS / Kainji hydro node,Transmission Node,9.86409,4.61241,330,kV,Niger,https://en.wikipedia.org/wiki/Kainji_Dam,High,"Major 330 kV hydro bus; northern backbone hub."
T_JEBBA,T_JEBBA,Jebba TS / Jebba hydro node,Transmission Node,9.1,5.0,330,kV,Kwara/Niger,https://en.wikipedia.org/wiki/Jebba_Hydroelectric_Power_Station,Medium,"Hydro cascade node between Kainji and Shiroro; confirm exact coordinates via snap."
T_SHIR,T_SHIR,Shiroro TS / Shiroro hydro node,Transmission Node,9.9724,6.8353,330,kV,Niger,https://en.wikipedia.org/wiki/Shiroro_Hydroelectric_Power_Station,High,"Northern hydro node; connected to Abuja/Kaduna corridors."
T_KAT,T_KAT,Katampe TS (Abuja),Transmission Node,9.11482,7.47673,330/132,kV,FCT Abuja,https://tcn.gov.ng/,Medium,"Katampe 330/132/33 kV station; approximated centroid from OSM/Mapcarta."
T_GEREGU,T_GEREGU,Geregu TS (Kogi),Transmission Node,7.563,6.6928,330/132,kV,Kogi,https://en.wikipedia.org/wiki/Geregu_Power_Station,High,"Central hub connecting Geregu generation to trunk."
T_EGBIN,T_EGBIN,Egbin TS (Lagos),Transmission Node,6.5632,3.6152,330/132,kV,Lagos,https://en.wikipedia.org/wiki/Egbin_Thermal_Power_Station,High,"Major south-west hub; Egbin generation/substation cluster."
T_OSOGBO,T_OSOGBO,Osogbo TS,Transmission Node,7.75960,4.57630,330/132,kV,Osun,https://tcn.gov.ng/,Medium,"Regional western corridor hub; approximated centroid from OSM."
T_PORTHR,T_PORTHR,Port Harcourt Main TS,Transmission Node,4.81420,7.05010,330/132,kV,Rivers,https://tcn.gov.ng/,Medium,"Key Delta/Niger Delta hub; approximated centroid from OSM."
T_SAPELE,T_SAPELE,Sapele TS,Transmission Node,5.9254,5.645,330/132,kV,Delta,https://en.wikipedia.org/wiki/Sapele_Power_Station,High,"Delta thermal/transmission hub."
T_AFAM,T_AFAM,Afam TS (Rivers),Transmission Node,4.83577,7.22704,330/132,kV,Rivers,https://en.wikipedia.org/wiki/Afam_Power_Station,Medium,"Afam power complex region; approximated centroid from OSM."

SS_KATAMPE,SS_KATAMPE,Katampe 330/132/33 kV TS,Substation,9.11482,7.47673,330/132/33,kV,FCT Abuja,https://tcn.gov.ng/,Medium,"Katampe substation (site centroid; same as T_KAT)."
SS_EGBIN,SS_EGBIN,Egbin 330/132 kV TS,Substation,6.5632,3.6152,330/132,kV,Lagos,https://en.wikipedia.org/wiki/Egbin_Thermal_Power_Station,High,"Egbin substation (Ikorodu/Lagos)."
SS_FUNTUA,SS_FUNTUA,Funtua 330/132 TS,Substation,11.52881,7.32020,330/132,kV,Katsina,https://tcn.gov.ng/,Medium,"Funtua substation (town centroid placeholder)."
SS_SAPELE,SS_SAPELE,Sapele 330/132 TS,Substation,5.9254,5.645,330/132,kV,Delta,https://en.wikipedia.org/wiki/Sapele_Power_Station,High,"Sapele substation."
SS_SHIRORO,SS_SHIRORO,Shiroro substation (near hydro plant),Substation,9.97,6.8353,330,kV,Niger,https://en.wikipedia.org/wiki/Shiroro_Hydroelectric_Power_Station,High,"Shiroro substation."
SS_JEBBA,SS_JEBBA,Jebba substation,Substation,9.1,5.0,330,kV,Kwara/Niger,https://en.wikipedia.org/wiki/Jebba_Hydroelectric_Power_Station,Medium,"Jebba substation (approx coords)."
SS_PORTHR,SS_PORTHR,Port Harcourt Main substation,Substation,4.81420,7.05010,330/132,kV,Rivers,https://tcn.gov.ng/,Medium,"Port Harcourt / Eleme substation cluster (site centroid)."

P_EGBIN,P_EGBIN,Egbin Thermal Power,Power Plant,6.5632,3.6152,1320,MW,Lagos,https://en.wikipedia.org/wiki/Egbin_Thermal_Power_Station,High,"Egbin thermal plant."
P_SHIR,P_SHIR,Shiroro Hydro,Power Plant,9.9724,6.8353,600,MW,Niger,https://en.wikipedia.org/wiki/Shiroro_Hydroelectric_Power_Station,High,"Shiroro hydro plant."
P_KAINJI,P_KAINJI,Kainji Hydro,Power Plant,9.86409,4.61241,760,MW,Niger,https://en.wikipedia.org/wiki/Kainji_Dam,High,"Kainji hydro plant."
P_JEBBA,P_JEBBA,Jebba Hydro,Power Plant,9.1,5.0,578,MW,Kwara/Niger,https://en.wikipedia.org/wiki/Jebba_Hydroelectric_Power_Station,Medium,"Jebba hydro plant (approx coords)."
P_GEREGU,P_GEREGU,Geregu Power Plant,Power Plant,7.563,6.6928,435,MW,Kogi,https://en.wikipedia.org/wiki/Geregu_Power_Station,High,"Geregu (gas-fired) plant."

I_DANGOTE,I_DANGOTE,Dangote Refinery & Fertiliser,Industrial Hub,6.459,4.021,,,Lagos,https://www.dangote.com/,High,"Refinery & fertilizer industrial demand hub."
I_AJAOKUTA,I_AJAOKUTA,Ajaokuta Steel Complex,Industrial Hub,7.50594,6.69245,,,Kogi,https://en.wikipedia.org/wiki/Ajaokuta_Steel_Mill,Medium,"Ajaokuta complex (partial/mothballed)."
I_INDR,I_INDR,Indorama / Eleme Petrochemical,Industrial Hub,4.8331,7.09511,,,Rivers,https://www.indoramafertilizers.com/,High,"Petrochemical / fertilizer industrial hub."
"""

# -------------------
# Helpers
# -------------------
def read_power_csv(path: Path) -> pd.DataFrame:
    if path.exists():
        try:
            df = pd.read_csv(path, dtype=str)
            log.info("Loaded power CSV: %s (%d rows)", path.name, len(df))
            return df
        except Exception as e:
            log.error("Failed reading %s: %s", path, e)
            return pd.DataFrame()
    else:
        log.info("Power CSV not found at %s — using built-in compiled asset list.", path)
        return pd.read_csv(io.StringIO(BACKBONE_DATA), dtype=str)


def normalize_power_df(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    mapping = {}
    for col in df.columns:
        c = col.lower().strip()
        if c in ("asset_id", "assetid", "id"):
            mapping[col] = "Asset_ID"
        elif c in ("node_id", "nodeid"):
            mapping[col] = "Node_ID"
        elif "name" in c:
            mapping[col] = "Name"
        elif c.startswith("type"):
            mapping[col] = "Type"
        elif re.search(r"\blat\b|latitude", c):
            mapping[col] = "Latitude"
        elif re.search(r"\blon\b|\blong\b|longitude", c):
            mapping[col] = "Longitude"
        elif "capacity" in c:
            mapping[col] = "Capacity"
        elif "unit" in c:
            mapping[col] = "Units"
        elif "state" in c or "region" in c:
            mapping[col] = "State_Region"
        elif "source" in c or "url" in c:
            mapping[col] = "Source_URL"
        elif "confidence" in c:
            mapping[col] = "Confidence"
        elif "note" in c:
            mapping[col] = "Notes"
    df = df.rename(columns=mapping)

    # Ensure canonical columns exist
    canonical = ["Asset_ID","Node_ID","Name","Type","Latitude","Longitude","Capacity","Units","State_Region","Source_URL","Confidence","Notes"]
    for c in canonical:
        if c not in df.columns:
            df[c] = None

    # If Asset_ID missing, derive from Node_ID or Name
    if df["Asset_ID"].isnull().all() or (df["Asset_ID"].astype(str).str.strip() == "").all():
        df["Asset_ID"] = df["Node_ID"].fillna(df["Name"]).fillna(method="ffill").astype(str).apply(lambda s: re.sub(r"[^A-Za-z0-9_]+","_",s)[:64])

    if df["Node_ID"].isnull().all() or (df["Node_ID"].astype(str).str.strip() == "").all():
        df["Node_ID"] = df["Asset_ID"]

    # Numeric coercion
    df["Latitude"] = pd.to_numeric(df["Latitude"], errors="coerce")
    df["Longitude"] = pd.to_numeric(df["Longitude"], errors="coerce")

    # trim whitespace in text fields
    for c in ["Asset_ID","Node_ID","Name","Type","Units","State_Region","Source_URL","Confidence","Notes"]:
        df[c] = df[c].astype(str).replace({"nan":""}).str.strip()

    return df[canonical]


def df_to_gdf(df: pd.DataFrame) -> gpd.GeoDataFrame:
    geoms = []
    for _, r in df.iterrows():
        lat = r["Latitude"]
        lon = r["Longitude"]
        if pd.notna(lat) and pd.notna(lon):
            geoms.append(Point(float(lon), float(lat)))
        else:
            geoms.append(None)
    gdf = gpd.GeoDataFrame(df.copy(), geometry=geoms, crs=CRS_WGS)
    return gdf


def dedupe_power_gdf(gdf: gpd.GeoDataFrame) -> gpd.GeoDataFrame:
    df = gdf.copy()
    
    # 1. Create a "Priority" column for sorting
    # We want Transmission Nodes to be kept over Power Plants or Substations at the same site
    def get_priority(t):
        t = str(t).lower()
        if "transmission" in t: return 1
        if "substation" in t: return 2
        if "plant" in t: return 3
        return 4
    
    df["_priority"] = df["Type"].apply(get_priority)
    df["_hasgeom"] = df.geometry.notnull()
    
    # 2. Sort so that the highest priority (lowest number) and with geometry comes first
    df = df.sort_values(by=["Asset_ID", "_hasgeom", "_priority"], ascending=[True, False, True])
    df = df.drop_duplicates(subset=["Asset_ID"], keep="first")

    # 3. Spatial deduplication (11m radius)
    def rcoord(x):
        try: return round(float(x), 4)
        except: return None
        
    df["_latr"] = df["Latitude"].apply(rcoord)
    df["_lonr"] = df["Longitude"].apply(rcoord)
    
    with_coords = df.dropna(subset=["_latr","_lonr"])
    without_coords = df[df[["_latr","_lonr"]].isnull().any(axis=1)]
    
    # Sort again to ensure the 'T_' node survives the spatial drop
    with_coords = with_coords.sort_values(by=["_priority", "_hasgeom"], ascending=[True, False])
    with_coords = with_coords.drop_duplicates(subset=["_latr","_lonr"], keep="first")
    
    out = pd.concat([with_coords, without_coords], ignore_index=True, sort=False)
    out = out.drop(columns=["_priority", "_hasgeom", "_latr", "_lonr"])
    
    return gpd.GeoDataFrame(out, geometry=out.geometry, crs=CRS_WGS)


def load_lines(local_path: Path) -> gpd.GeoDataFrame:
    # prefer local
    if local_path.exists():
        try:
            lines = gpd.read_file(local_path)
            if lines.crs is None:
                lines.set_crs(CRS_WGS, inplace=True)
            lines = lines[lines.geometry.notnull() & lines.geometry.type.isin(["LineString","MultiLineString"])]
            log.info("Loaded local lines: %s (%d features)", local_path.name, len(lines))
            return gpd.GeoDataFrame(lines.copy(), geometry="geometry", crs=CRS_WGS)
        except Exception as e:
            log.warning("Failed reading local lines: %s", e)

    # OSM fallback (power lines)
    try:
        import osmnx as ox
        log.info("Fetching power lines from OSM (this may take a minute)...")
        try:
            gdf = ox.geometries_from_place("Nigeria", tags={"power":"line"})
        except Exception:
            gdf = ox.features_from_place("Nigeria", tags={"power":"line"})
        
        if gdf is None or gdf.empty:
            log.warning("OSM returned no power lines.")
            return gpd.GeoDataFrame({"geometry":[]}, geometry="geometry", crs=CRS_WGS)
        
        # Filter for valid line geometries
        gdf = gdf[gdf.geometry.notnull() & gdf.geometry.type.isin(["LineString","MultiLineString"])]
        log.info("Fetched %d power-line features from OSM", len(gdf))

        # --- NEW: SAVE LOCALLY TO PREVENT RE-DOWNLOADING ---
        try:
            # We reset_index because OSM data often has a complex MultiIndex 
            # that GeoJSON/Fiona cannot handle easily.
            save_gdf = gdf.reset_index()
            # Ensure the directory exists
            local_path.parent.mkdir(parents=True, exist_ok=True)
            save_gdf.to_file(local_path, driver='GeoJSON')
            log.info("Saved OSM lines to %s for faster future loading.", local_path)
        except Exception as save_error:
            log.warning("Could not save OSM lines locally: %s", save_error)
        # --------------------------------------------------

        return gpd.GeoDataFrame(gdf.copy(), geometry="geometry", crs=CRS_WGS)

    except Exception as e:
        log.warning("OSM fetch failed (osmnx missing or network). Returning empty lines. Error: %s", e)
        return gpd.GeoDataFrame({"geometry":[]}, geometry="geometry", crs=CRS_WGS)


# -------------------
# Main routine
# -------------------
def main():
    # 1) Read and normalise power CSV (or fallback compiled list)
    raw_df = read_power_csv(POWER_CSV)
    pow_norm = normalize_power_df(raw_df) if not raw_df.empty else normalize_power_df(pd.read_csv(io.StringIO(BACKBONE_DATA), dtype=str))
    gdf = df_to_gdf(pow_norm)
    log.info("Total records loaded (power): %d; with coords: %d", len(gdf), gdf.geometry.notnull().sum())

    # 2) dedupe
    gdf = dedupe_power_gdf(gdf)
    log.info("After deduplication: %d records (with coords: %d)", len(gdf), gdf.geometry.notnull().sum())

    # 3) load line geometries
    lines = load_lines(LOCAL_LINES)
    if lines.empty:
        log.warning("No line geometries available — snapping will be skipped.")
    else:
        try:
            lines = lines.explode(index_parts=False).reset_index(drop=True)
        except Exception:
            lines = lines.reset_index(drop=True)

    # 4) snap to lines (if available) using metric CRS
    snapped = gdf.copy()
    if not lines.empty:
        lines_m = lines.to_crs(CRS_METRIC)
        snapped_m = snapped.to_crs(CRS_METRIC)
        sindex = lines_m.sindex
        snapped_geoms = []
        snap_dists = []
        nearest_line_idx = []
        for idx, row in snapped_m.iterrows():
            pt = row.geometry
            if pt is None or pt.is_empty:
                snapped_geoms.append(None)
                snap_dists.append(np.nan)
                nearest_line_idx.append(None)
                continue
            candidates = list(sindex.intersection(pt.bounds))
            if candidates:
                subset = lines_m.iloc[candidates]
                d = subset.geometry.distance(pt)
                i_min = d.idxmin()
                min_dist = float(d.min())
                nearest_geometry = subset.loc[i_min].geometry
            else:
                d = lines_m.geometry.distance(pt)
                i_min = d.idxmin()
                min_dist = float(d.min())
                nearest_geometry = lines_m.loc[i_min].geometry
            snapped_pt = nearest_geometry.interpolate(nearest_geometry.project(pt))
            snapped_geoms.append(snapped_pt)
            snap_dists.append(min_dist)
            nearest_line_idx.append(str(i_min))
        snapped_m["geometry"] = snapped_geoms
        snapped_m["snap_dist_m"] = snap_dists
        snapped_m["nearest_line_id"] = nearest_line_idx
        snapped = snapped_m.to_crs(CRS_WGS)
        log.info("Snapped %d points to lines.", snapped.geometry.notnull().sum())
    else:
        snapped["snap_dist_m"] = np.nan
        snapped["nearest_line_id"] = None

    # 5) Build small topology & compute centrality
    node_geom = {}
    for _, r in snapped.dropna(subset=["geometry"]).iterrows():
        node_geom[str(r["Node_ID"])] = r.geometry

    provisional_edges = [
        ("T_KAINJI","T_JEBBA"),
        ("T_JEBBA","T_SHIR"),
        ("T_SHIR","T_KAT"),
        ("T_KAT","T_GEREGU"),
        ("T_GEREGU","T_EGBIN"),
        ("T_EGBIN","T_SAPELE"),
        ("T_SAPELE","T_PORTHR"),
    ]

    G = nx.Graph()
    for nid in node_geom.keys():
        G.add_node(nid)

    for u,v in provisional_edges:
        if u in node_geom and v in node_geom:
            try:
                up = gpd.GeoSeries([node_geom[u]], crs=CRS_WGS).to_crs(CRS_METRIC).iloc[0]
                vp = gpd.GeoSeries([node_geom[v]], crs=CRS_WGS).to_crs(CRS_METRIC).iloc[0]
                dist = up.distance(vp)
                w = float(dist)
            except Exception:
                # fallback to haversine
                def hav_m(a,b):
                    import math
                    lon1, lat1 = a.x, a.y
                    lon2, lat2 = b.x, b.y
                    R = 6371000.0
                    phi1 = math.radians(lat1); phi2 = math.radians(lat2)
                    dphi = math.radians(lat2-lat1); dl = math.radians(lon2-lon1)
                    aa = math.sin(dphi/2)**2 + math.cos(phi1)*math.cos(phi2)*math.sin(dl/2)**2
                    return 2*R*math.asin(min(1, math.sqrt(aa)))
                w = hav_m(node_geom[u], node_geom[v])
            G.add_edge(u,v,weight=w)
        else:
            log.debug("Provisional edge skipped (missing node): %s - %s", u, v)

    if G.number_of_nodes() == 0:
        log.warning("No graph nodes present; centrality skipped.")
        snapped["betweenness_hub_score"] = 0.0
    else:
        try:
            btw = nx.betweenness_centrality(G, weight="weight", normalized=True)
        except Exception:
            btw = nx.betweenness_centrality(G, normalized=True)
        snapped["betweenness_hub_score"] = snapped["Node_ID"].map(btw).fillna(0.0)

    # 6) Exposure from Zenodo attacks: count events within 10km
    if not Path(ZENODO_EVENTS).exists():
        log.warning("Zenodo events CSV not found at %s — skipping exposure.", ZENODO_EVENTS)
        snapped["attacks_within_10km"] = np.nan
    else:
        try:
            evdf = pd.read_csv(ZENODO_EVENTS, dtype=str)
            lat_cols = [c for c in evdf.columns if re.search(r'\blat(itude)?\b', c.lower())]
            lon_cols = [c for c in evdf.columns if re.search(r'\blon(gitude)?\b', c.lower())]
            if not lat_cols or not lon_cols:
                log.error("Zenodo file missing lat/lon columns; exposure skipped.")
                snapped["attacks_within_10km"] = np.nan
            else:
                evdf["latitude"] = pd.to_numeric(evdf[lat_cols[0]], errors="coerce")
                evdf["longitude"] = pd.to_numeric(evdf[lon_cols[0]], errors="coerce")
                evdf = evdf.dropna(subset=["latitude","longitude"])
                evg = gpd.GeoDataFrame(evdf, geometry=gpd.points_from_xy(evdf.longitude, evdf.latitude), crs=CRS_WGS)
                evm = evg.to_crs(CRS_METRIC)
                nodes_m = snapped.to_crs(CRS_METRIC)
                counts = []
                for _, a in nodes_m.iterrows():
                    if a.geometry is None or a.geometry.is_empty:
                        counts.append(np.nan)
                        continue
                    d = evm.geometry.distance(a.geometry)
                    counts.append(int((d <= 10000).sum()))
                snapped["attacks_within_10km"] = counts
                log.info("Computed attacks-within-10km for %d nodes.", len(snapped))
        except Exception as e:
            log.error("Failed computing exposure: %s", e)
            snapped["attacks_within_10km"] = np.nan

    # 7) export results
    try:
        snapped.to_file(OUTPUT_GPKG, layer=OUTPUT_LAYER, driver="GPKG")
        log.info("Wrote GeoPackage: %s (layer=%s)", OUTPUT_GPKG, OUTPUT_LAYER)
    except Exception as e:
        log.error("Failed writing GPKG: %s", e)

    try:
        # Save as CSV without the geometry column
        snapped.drop(columns="geometry").to_csv(EXPOSURE_CSV, index=False)
        log.info(f"Wrote exposure CSV: {EXPOSURE_CSV}")
    except Exception as e:
        log.error(f"Failed writing exposure CSV: {e}")

    # summary
    log.info("Final: %d records (with coords: %d). Graph: %d nodes, %d edges",
             len(snapped), snapped.geometry.notnull().sum(), G.number_of_nodes(), G.number_of_edges())


if __name__ == "__main__":
    main()

INFO: Power CSV not found at data/raw/power_sector_assets.csv — using built-in compiled asset list.
INFO: Total records loaded (power): 25; with coords: 25
INFO: After deduplication: 15 records (with coords: 15)
INFO: Fetching power lines from OSM (this may take a minute)...
/Users/shil6591/miniconda3/envs/chapter1_env/lib/python3.10/site-packages/osmnx/_overpass.py:271: UserWarning: This area is 429 times your configured Overpass max query area size. It will automatically be divided up into multiple sub-queries accordingly. This may take a long time.
  multi_poly_proj = utils_geo._consolidate_subdivide_geometry(poly_proj)
INFO: Fetched 1254 power-line features from OSM
INFO: Created 1,254 records
INFO: Saved OSM lines to data/raw/nigeria_transmission_lines.geojson for faster future loading.
INFO: Snapped 15 points to lines.
INFO: Computed attacks-within-10km for 15 nodes.
INFO: Created 15 records
INFO: Wrote GeoPackage: data/processed/nigeria_power_topology.gpkg (layer=power_nodes)
IN